# Tech Challenge - Fase 3

## Base estruturada de pacientes

Nesta etapa será criada uma base de dados SQLite contendo registros clínicos sintéticos.

A base será utilizada pelo assistente médico para consultar informações atualizadas do paciente antes da geração da resposta.

O banco contém apenas dados sintéticos e será integrado posteriormente ao pipeline com LangChain e LangGraph.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import sqlite3

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

DATA_DIR = PROJECT_DIR / "data"

PROCESSED_DIR = DATA_DIR / "processed"

DATABASE_DIR = DATA_DIR / "database"

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CSV_PATH = PROCESSED_DIR / "pacientes_processados.csv"

DB_PATH = DATABASE_DIR / "hospital.db"

print("CSV:", CSV_PATH)
print("Banco:", DB_PATH)

print("CSV existe:", CSV_PATH.exists())

CSV: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/pacientes_processados.csv
Banco: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/database/hospital.db
CSV existe: True


In [ ]:
df_pacientes = pd.read_csv(CSV_PATH)

print("Quantidade de pacientes:", len(df_pacientes))
print()
print(df_pacientes.columns.tolist())

df_pacientes.head()

Quantidade de pacientes: 5

['patient_id', 'idade', 'sexo', 'diagnostico', 'glicemia_mg_dl', 'hba1c_percentual', 'pressao_sistolica', 'pressao_diastolica', 'imc', 'creatinina_mg_dl', 'colesterol_total_mg_dl', 'exame_pendente']


,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,Sem exame pendente
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,Sem exame pendente
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
conexao = sqlite3.connect(DB_PATH)

df_pacientes.to_sql(
    "pacientes",
    conexao,
    if_exists="replace",
    index=False
)

conexao.commit()

print("Tabela 'pacientes' criada com sucesso.")

Tabela 'pacientes' criada com sucesso.


In [ ]:
consulta_tabelas = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql_query(
    consulta_tabelas,
    conexao
)

,name
0,pacientes


In [ ]:
consulta = """
SELECT *
FROM pacientes;
"""

df_banco = pd.read_sql_query(
    consulta,
    conexao
)

df_banco

,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,Sem exame pendente
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,Sem exame pendente
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
def buscar_paciente(patient_id):
    consulta = """
    SELECT *
    FROM pacientes
    WHERE patient_id = ?
    """

    resultado = pd.read_sql_query(
        consulta,
        conexao,
        params=(patient_id,)
    )

    if resultado.empty:
        return None

    return resultado.iloc[0].to_dict()

In [ ]:
paciente = buscar_paciente("PAC001")

paciente

{'patient_id': 'PAC001',
 'idade': 52,
 'sexo': 'F',
 'diagnostico': 'Diabetes Mellitus Tipo 2',
 'glicemia_mg_dl': 205,
 'hba1c_percentual': 9.2,
 'pressao_sistolica': 145,
 'pressao_diastolica': 95,
 'imc': 31.2,
 'creatinina_mg_dl': 1.0,
 'colesterol_total_mg_dl': 218,
 'exame_pendente': 'Microalbuminúria'}

In [ ]:
paciente_inexistente = buscar_paciente("PAC999")

print(paciente_inexistente)

None


In [ ]:
def formatar_contexto_paciente(paciente):

    if paciente is None:
        return "Paciente não encontrado."

    return (
        f"Paciente {paciente['patient_id']}, "
        f"{paciente['idade']} anos, "
        f"sexo {paciente['sexo']}. "
        f"Diagnóstico: {paciente['diagnostico']}. "
        f"Glicemia: {paciente['glicemia_mg_dl']} mg/dL. "
        f"HbA1c: {paciente['hba1c_percentual']}%. "
        f"Pressão arterial: "
        f"{paciente['pressao_sistolica']}/"
        f"{paciente['pressao_diastolica']} mmHg. "
        f"IMC: {paciente['imc']}. "
        f"Creatinina: {paciente['creatinina_mg_dl']} mg/dL. "
        f"Colesterol total: "
        f"{paciente['colesterol_total_mg_dl']} mg/dL. "
        f"Exame pendente: "
        f"{paciente['exame_pendente']}."
    )

In [ ]:
paciente = buscar_paciente("PAC001")

contexto = formatar_contexto_paciente(
    paciente
)

print(contexto)

Paciente PAC001, 52 anos, sexo F. Diagnóstico: Diabetes Mellitus Tipo 2. Glicemia: 205 mg/dL. HbA1c: 9.2%. Pressão arterial: 145/95 mmHg. IMC: 31.2. Creatinina: 1.0 mg/dL. Colesterol total: 218 mg/dL. Exame pendente: Microalbuminúria.


In [ ]:
for patient_id in [
    "PAC001",
    "PAC002",
    "PAC003",
    "PAC004",
    "PAC005"
]:
    paciente = buscar_paciente(patient_id)

    print(
        formatar_contexto_paciente(paciente)
    )

    print("-" * 80)

Paciente PAC001, 52 anos, sexo F. Diagnóstico: Diabetes Mellitus Tipo 2. Glicemia: 205 mg/dL. HbA1c: 9.2%. Pressão arterial: 145/95 mmHg. IMC: 31.2. Creatinina: 1.0 mg/dL. Colesterol total: 218 mg/dL. Exame pendente: Microalbuminúria.
--------------------------------------------------------------------------------
Paciente PAC002, 46 anos, sexo M. Diagnóstico: Diabetes Mellitus Tipo 2. Glicemia: 118 mg/dL. HbA1c: 6.7%. Pressão arterial: 128/82 mmHg. IMC: 27.4. Creatinina: 0.9 mg/dL. Colesterol total: 185 mg/dL. Exame pendente: Sem exame pendente.
--------------------------------------------------------------------------------
Paciente PAC003, 63 anos, sexo F. Diagnóstico: Diabetes Mellitus Tipo 2. Glicemia: 172 mg/dL. HbA1c: 8.3%. Pressão arterial: 138/88 mmHg. IMC: 29.8. Creatinina: 1.3 mg/dL. Colesterol total: 201 mg/dL. Exame pendente: Fundo de olho.
--------------------------------------------------------------------------------
Paciente PAC004, 39 anos, sexo M. Diagnóstico: Diabet

In [ ]:
conexao.close()

print("Conexão com o banco encerrada.")

Conexão com o banco encerrada.
